## 1. Mount Google Drive

Mount your Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls

drive  sample_data


In [ ]:
import os

In [ ]:
os.chdir('/content/drive/MyDrive/Biomarker_Project')

In [ ]:
!ls

bert_tokenizer		    FDA_therapy_biomarker_extraction.ipynb
biomarker_nlp		    Final_Biomarker_Report.csv
biomarker_nlp_1		    Final_Biomarker_Report.gsheet
biomarker_output.csv	    negCue
biomarker_output.gsheet     negScope
biomarker_report_final.csv  NLP_Biomarker_Extraction.ipynb
checkpoint_1.csv	    therapy_links


In [ ]:
import os

In [ ]:
os.chdir('/content/drive/MyDrive/Biomarker_Project')

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

def get_nci_drug_links():
    url = "https://www.cancer.gov/about-cancer/treatment/drugs/cancer-drugs"
    base_url = "https://www.cancer.gov"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    print(f"Fetching drug directory from: {url}")

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        drug_links = []
        content_area = soup.find('article') or soup.find('main')

        if not content_area:
            print("Error finding the main content area.")
            return pd.DataFrame()

        # Find all <a> tags within the content area
        all_links = content_area.find_all('a', href=True)

        for link in tqdm(all_links, desc="Filtering drug links"):
            href = link['href']
            text = link.text.strip()

            # Filter for links that go to specific drug pages
            # (usually under /about-cancer/treatment/drugs/)
            if "/about-cancer/treatment/drugs/" in href and text:
                # Ensure it's not the main landing page link
                if href != "/about-cancer/treatment/drugs/cancer-drugs":
                    drug_links.append(base_url + href if href.startswith('/') else href)

        return drug_links
    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return pd.DataFrame()

# Run the scraper
df_drug_list = get_nci_drug_links()


Fetching drug directory from: https://www.cancer.gov/about-cancer/treatment/drugs/cancer-drugs


Filtering drug links: 100%|██████████| 775/775 [00:00<00:00, 226049.07it/s]


In [ ]:
len(df_drug_list)

748

In [ ]:
!ls

bert_tokenizer		    FDA_therapy_biomarker_extraction.ipynb
biomarker_nlp		    Final_Biomarker_Report.csv
biomarker_nlp_1		    Final_Biomarker_Report.gsheet
biomarker_output.csv	    negCue
biomarker_output.gsheet     negScope
biomarker_report_final.csv  NLP_Biomarker_Extraction.ipynb
checkpoint_1.csv	    therapy_links


In [ ]:
with open('therapy_extraction.txt','w') as file:
  file .write(','.join(df_drug_list))